<a href="https://colab.research.google.com/github/imabigger/my_DS_recipe_book/blob/main/%EB%AA%A8%EA%B8%B0%EB%B9%84%ED%96%89%EA%B2%BD%EB%A1%9C%EC%98%88%EC%B8%A1/%EB%AA%A8%EA%B8%B0%EB%B9%84%ED%96%89%EA%B6%A4%EC%A0%814_T_%EB%8D%94_%EC%9E%90%EC%84%B8%ED%9E%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!cp "/content/drive/MyDrive/ai/모기비행궤적예측ai경진대회/open.zip" "/content/"
!unzip -q "/content/open.zip" -d "/content/"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
import os
import glob
from tqdm import tqdm

class DynamicFeatureExtractor:
    def __init__(self, feature_list=None):
        self.feature_list = feature_list or ['log_kappa', 'v_norm', 'T']
        self.dt = 0.04
        self.eps = 1e-8

    def __call__(self, x_raw):
        features_seq = []
        historical_seq = []

        # 연속된 프레임 간의 계산을 위해 기저들을 저장할 리스트
        T_list, N_list, B_list = [], [], []
        v_norms = []
        kappas = []

        # 1차 패치 루프: 기본 프레네 기저 및 곡률 계산
        for i in range(2, 11):
            p0, p_m1, p_m2 = x_raw[i], x_raw[i-1], x_raw[i-2]
            v = (p0 - p_m1) / self.dt
            a = (p0 - 2 * p_m1 + p_m2) / (self.dt ** 2)

            v_norm = np.linalg.norm(v) + self.eps
            v_norms.append(v_norm)
            cross_va = np.cross(v, a)
            cross_norm = np.linalg.norm(cross_va) + self.eps

            T_vec = v / v_norm
            if cross_norm > self.eps:
                kappa = cross_norm / (v_norm ** 3)
                B_vec = cross_va / cross_norm
                N_vec = np.cross(B_vec, T_vec)
            else:
                kappa = 0.0
                B_vec = np.array([0.0, 0.0, 0.0])
                N_vec = np.array([0.0, 0.0, 0.0])

            T_list.append(T_vec)
            N_list.append(N_vec)
            B_list.append(B_vec)
            kappas.append(np.clip(kappa, 0.0, 1000.0))

        R_current = np.column_stack([T_list[-1], N_list[-1], B_list[-1]])

        # 2차 패치 루프: 고차 기하 피처(Torsion, M_matrix) 조립
        for idx in range(9):
            i = idx + 2
            kappa = kappas[idx]
            log_kappa = np.log(kappa + self.eps)
            v_norm = v_norms[idx]
            R_curr = np.column_stack([T_list[idx], N_list[idx], B_list[idx]])

            # Torsion 근사 계산
            if idx > 0:
                dt_disp = v_norm * self.dt
                torsion = np.dot(B_list[idx-1] - B_list[idx], N_list[idx]) / (dt_disp + self.eps)
                # 프레임 전이 행렬 M_i = R_{i-1}^T * R_i
                R_prev = np.column_stack([T_list[idx-1], N_list[idx-1], B_list[idx-1]])
                M_matrix = np.dot(R_prev.T, R_curr)
            else:
                torsion = 0.0
                M_matrix = np.eye(3)

            vec_M = M_matrix.flatten() # 9차원 벡터화

            # 조건부 타겟 추출 기하 연산
            if i + 2 <= 10:
                delta_local = np.dot(R_curr.T, x_raw[i+2] - x_raw[i])
            else:
                delta_local = np.array([np.nan, np.nan, np.nan])
            historical_seq.append(delta_local)

            # 동적 피처 빌드
            step_feat = []
            for f in self.feature_list:
                if f == 'log_kappa': step_feat.append(log_kappa)
                elif f == 'v_norm': step_feat.append(v_norm)
                elif f == 'torsion': step_feat.append(torsion)
                elif f == 'vec_M': step_feat.extend(vec_M.tolist())
                elif f == 'T': step_feat.extend(T_list[idx].tolist())
            features_seq.append(step_feat)

        return (
            torch.tensor(features_seq, dtype=torch.float32),
            torch.tensor(historical_seq, dtype=torch.float32),
            R_current
        )

class MosquitoFrenetDataset(Dataset):
    def __init__(self, data_dir, label_path, feature_list=None):
        self.file_paths = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
        self.labels_df = pd.read_csv(label_path)

        if 'id' in self.labels_df.columns:
            self.labels_df.set_index('id', inplace=True)

        self.dt = 0.04
        self.extractor = DynamicFeatureExtractor(feature_list)

        self.precomputed_data = []

        self.historical_dp_T = []
        self.historical_dp_N = []
        self.historical_dp_B = []

        print("데이터 및 피처 사전 계산 중...")
        for file_path in tqdm(self.file_paths):
            df = pd.read_csv(file_path)
            x_raw = df[['x', 'y', 'z']].values[:11, :]
            file_name = os.path.basename(file_path).split('.')[0]

            # 피처 추출기에서 피처 배열과 시점별 과거 타겟 배열을 동시에 받습니다.
            features_tensor, historical_tensor, R_current = self.extractor(x_raw)

            # NaN을 필터링하여 시각화용 데이터 리스트를 구성합니다.
            valid_hist = historical_tensor[~torch.isnan(historical_tensor[:, 0])].numpy()
            if len(valid_hist) > 0:
                self.historical_dp_T.extend(valid_hist[:, 0])
                self.historical_dp_N.extend(valid_hist[:, 1])
                self.historical_dp_B.extend(valid_hist[:, 2])

            y_abs = self.labels_df.loc[file_name, ['x', 'y', 'z']].values.astype(np.float32)
            p0_final = x_raw[-1]

            delta_global = y_abs - p0_final
            target_local = np.dot(R_current.T, delta_global)

            self.precomputed_data.append({
                'features': features_tensor,                    # 형태: (9, num_features)
                'historical_targets': historical_tensor,        # 형태: (9, 3) 9패치에 대한 80ms 후 타겟 마지막 2개는 NAN
                'target_local': torch.tensor(target_local, dtype=torch.float32),
                'target_global': torch.tensor(delta_global, dtype=torch.float32),
                'R_matrix': torch.from_numpy(R_current).float()
            })

    def __len__(self):
        return len(self.precomputed_data)

    def __getitem__(self, idx):
        return self.precomputed_data[idx]


train_data_dir = "/content/train"
train_label_path = "/content/train_labels.csv"
test_data_dir = '/content/test'

# 하이퍼파라미터 설정
batch_size = 1024

dataset = MosquitoFrenetDataset(train_data_dir, train_label_path)

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
print(f"총 {len(dataset)}개의 훈련 데이터를 불러왔습니다.")

데이터 및 피처 사전 계산 중...


100%|██████████| 10000/10000 [00:32<00:00, 307.64it/s]

총 10000개의 훈련 데이터를 불러왔습니다.


In [ ]:
def plot_interactive_distributions(dataset):
    """
    plotly를 활용하여 과거 80ms 증강 분포와 마지막 시점 정답 분포를
    대화형 3D 산점도로 비교하여 렌더링합니다.
    """
    # 1. 과거 데이터 추출
    hist_T = np.array(dataset.historical_dp_T)
    hist_N = np.array(dataset.historical_dp_N)
    hist_B = np.array(dataset.historical_dp_B)

    if len(hist_T) == 0:
        print("데이터셋에 저장된 과거 변위 데이터가 없습니다.")
        return

    # 2. 마지막 시점 정답 타겟 추출
    targets = np.array([item['target_local'].numpy() for item in dataset.precomputed_data])
    gt_T = targets[:, 0]
    gt_N = targets[:, 1]
    gt_B = targets[:, 2]

    # 샘플링 보조 함수: 렌더링 부하를 줄이기 위해 모드별로 점을 샘플링합니다.
    def get_sampled_indices(T_arr, max_pts=10000):
        mode1_mask = T_arr > 0.1
        mode2_mask = T_arr <= 0.1

        idx1 = np.where(mode1_mask)[0]
        idx2 = np.where(mode2_mask)[0]

        if len(idx1) > max_pts: idx1 = np.random.choice(idx1, max_pts, replace=False)
        if len(idx2) > max_pts: idx2 = np.random.choice(idx2, max_pts, replace=False)

        return idx1, idx2

    hist_idx1, hist_idx2 = get_sampled_indices(hist_T)
    gt_idx1, gt_idx2 = get_sampled_indices(gt_T)

    # 1행 2열의 대화형 서브플롯 생성
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
        subplot_titles=('과거 80ms 증강 분포 (Historical)', '마지막 시점 정답 분포 (Ground Truth)')
    )

    # 왼쪽 플롯: 과거 증강 분포 (크기를 작게 설정)
    fig.add_trace(go.Scatter3d(
        x=hist_T[hist_idx1], y=hist_N[hist_idx1], z=hist_B[hist_idx1],
        mode='markers', marker=dict(size=1.5, color='blue', opacity=0.4),
        name='Hist Mode 1 (T > 0.1)'
    ), row=1, col=1)

    fig.add_trace(go.Scatter3d(
        x=hist_T[hist_idx2], y=hist_N[hist_idx2], z=hist_B[hist_idx2],
        mode='markers', marker=dict(size=1.5, color='red', opacity=0.4),
        name='Hist Mode 2 (T <= 0.1)'
    ), row=1, col=1)

    # 오른쪽 플롯: 정답 타겟 분포
    fig.add_trace(go.Scatter3d(
        x=gt_T[gt_idx1], y=gt_N[gt_idx1], z=gt_B[gt_idx1],
        mode='markers', marker=dict(size=2, color='blue', opacity=0.6),
        name='GT Mode 1 (T > 0.1)'
    ), row=1, col=2)

    fig.add_trace(go.Scatter3d(
        x=gt_T[gt_idx2], y=gt_N[gt_idx2], z=gt_B[gt_idx2],
        mode='markers', marker=dict(size=2, color='red', opacity=0.6),
        name='GT Mode 2 (T <= 0.1)'
    ), row=1, col=2)

    # 레이아웃 및 축 설정
    fig.update_layout(
        title='로컬 프레네(T-N-B) 변위 분포 비교 (Interactive)',
        height=700, width=1400,
        margin=dict(l=0, r=0, b=0, t=50)
    )

    # 두 장면의 축 이름을 동일하게 지정합니다.
    fig.update_scenes(
        xaxis_title='T (Tangent)',
        yaxis_title='N (Normal)',
        zaxis_title='B (Binormal)'
    )

    fig.show()

plot_interactive_distributions(dataset)

In [ ]:
from torch.utils.data import random_split
class FrenetVerificationLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, num_layers=1)
        # 최종 시점(Sequence의 마지막 9번째)의 hidden state를 받아 N, B 변위(2차원) 예측
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 2) # Output: [Delta_N, Delta_B]
        )

    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        # 마지막 타임스텝의 hidden state 추출
        last_hidden = lstm_out[:, -1, :]
        return self.regressor(last_hidden)

def run_strict_verification_experiment(feature_list, train_dir, label_path, epochs=1000, batch_size=1024, patience=30):
    # 1. 데이터셋 로드 및 분할 (8:2)
    full_dataset = MosquitoFrenetDataset(train_dir, label_path, feature_list=feature_list)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    input_dim = full_dataset[0]['features'].shape[1]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = FrenetVerificationLSTM(input_dim=input_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    best_val_loss = float('inf')
    patience_counter = 0  # 성능 정체 횟수를 기록할 카운터
    best_model_state = None # 최적의 모델 가중치를 저장할 변수

    print(f"조기 종료가 활성화된 검증 학습 시작 (Patience: {patience})")
    for epoch in range(epochs):
        # 학습 단계
        model.train()
        train_loss = 0.0
        for batch in train_dataloader:
            features = batch['features'].to(device)
            target_nb = batch['target_local'].to(device)[:, 1:3]

            optimizer.zero_grad()
            pred_nb = model(features)
            loss = criterion(pred_nb, target_nb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * features.size(0)

        # 검증 단계
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_dataloader:
                features = batch['features'].to(device)
                target_nb = batch['target_local'].to(device)[:, 1:3]
                pred_nb = model(features)
                loss = criterion(pred_nb, target_nb)
                val_loss += loss.item() * features.size(0)

        epoch_train_loss = train_loss / train_size
        epoch_val_loss = val_loss / val_size

        # 조기 종료 조건 검증 및 최적 모델 가중치 업데이트
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            patience_counter = 0  # 성능이 갱신되었으므로 카운터 초기화
            best_model_state = model.state_dict() # 최적 상태 저장
        else:
            patience_counter += 1  # 성능 개선이 없으면 카운터 누적

        # 10 에포크 주기 혹은 조기 종료 시점에 로그 출력
        if (epoch + 1) % 10 == 0 or epoch == 0 or patience_counter >= patience:
            print(f"Epoch [{epoch+1}/{epochs}] | Train MSE: {epoch_train_loss:.6f} | Val MSE: {epoch_val_loss:.6f} | Stagnation: {patience_counter}/{patience}")

        # 정체 횟수가 허용치(Patience)에 도달하면 루프 탈출
        if patience_counter >= patience:
            print(f"Early Stopping 트리거: {epoch+1} 에포크에서 학습이 자동 종료되었습니다.")
            break

    print(f"검증 완료 - 최적의 Validation MSE: {best_val_loss:.6f}")

    # 학습이 끝난 모델에 최적의 가중치를 다시 로드하여 반환
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return best_val_loss

In [ ]:
from torch._C import NoneType
print("\n=== 실험 1: 베이스라인 (v_norm) ===")
loss_baseline = run_strict_verification_experiment(
    feature_list=['v_norm'],
    train_dir=train_data_dir,
    label_path=train_label_path,
    epochs=1000
)

# 실험 2: 미분기하 정보(곡률, 비틀림, 기저 전이행렬)를 모두 주었을 때
print("\n=== 실험 2: 제안 기법 (v_norm + log_kappa + torsion + vec_M) ===")
loss_proposed = run_strict_verification_experiment(
    feature_list=['v_norm', 'log_kappa', 'torsion', 'vec_M'],
    train_dir=train_data_dir,
    label_path=train_label_path,
    epochs=1000
)

print("\n=== 실험 3: T 기법 (v_norm + log_kappa + T) ===")
loss_proposed = run_strict_verification_experiment(
    feature_list= ['log_kappa', 'v_norm', 'T'],
    train_dir=train_data_dir,
    label_path=train_label_path,
    epochs=1000
)


=== 실험 1: 베이스라인 (v_norm) ===
데이터 및 피처 사전 계산 중...


100%|██████████| 10000/10000 [00:30<00:00, 323.29it/s]


조기 종료가 활성화된 검증 학습 시작 (Patience: 15)
Epoch [1/1000] | Train MSE: 0.002071 | Val MSE: 0.000176 | Stagnation: 0/15
Epoch [10/1000] | Train MSE: 0.000113 | Val MSE: 0.000115 | Stagnation: 0/15
Epoch [20/1000] | Train MSE: 0.000112 | Val MSE: 0.000115 | Stagnation: 1/15
Epoch [30/1000] | Train MSE: 0.000112 | Val MSE: 0.000115 | Stagnation: 2/15
Epoch [40/1000] | Train MSE: 0.000112 | Val MSE: 0.000114 | Stagnation: 0/15
Epoch [50/1000] | Train MSE: 0.000112 | Val MSE: 0.000114 | Stagnation: 0/15
Epoch [60/1000] | Train MSE: 0.000112 | Val MSE: 0.000114 | Stagnation: 4/15
Epoch [70/1000] | Train MSE: 0.000112 | Val MSE: 0.000114 | Stagnation: 2/15
Epoch [80/1000] | Train MSE: 0.000112 | Val MSE: 0.000114 | Stagnation: 0/15
Epoch [90/1000] | Train MSE: 0.000112 | Val MSE: 0.000114 | Stagnation: 0/15
Epoch [100/1000] | Train MSE: 0.000112 | Val MSE: 0.000114 | Stagnation: 4/15
Epoch [110/1000] | Train MSE: 0.000111 | Val MSE: 0.000114 | Stagnation: 6/15
Epoch [120/1000] | Train MSE: 0.000112 

100%|██████████| 10000/10000 [00:30<00:00, 324.86it/s]


조기 종료가 활성화된 검증 학습 시작 (Patience: 15)
Epoch [1/1000] | Train MSE: 0.006154 | Val MSE: 0.001786 | Stagnation: 0/15
Epoch [10/1000] | Train MSE: 0.000152 | Val MSE: 0.000143 | Stagnation: 0/15
Epoch [20/1000] | Train MSE: 0.000127 | Val MSE: 0.000121 | Stagnation: 0/15
Epoch [30/1000] | Train MSE: 0.000120 | Val MSE: 0.000116 | Stagnation: 1/15
Epoch [40/1000] | Train MSE: 0.000115 | Val MSE: 0.000111 | Stagnation: 1/15
Epoch [50/1000] | Train MSE: 0.000112 | Val MSE: 0.000109 | Stagnation: 0/15
Epoch [60/1000] | Train MSE: 0.000111 | Val MSE: 0.000107 | Stagnation: 0/15
Epoch [70/1000] | Train MSE: 0.000109 | Val MSE: 0.000106 | Stagnation: 0/15
Epoch [80/1000] | Train MSE: 0.000108 | Val MSE: 0.000106 | Stagnation: 1/15
Epoch [90/1000] | Train MSE: 0.000107 | Val MSE: 0.000106 | Stagnation: 4/15
Epoch [100/1000] | Train MSE: 0.000106 | Val MSE: 0.000106 | Stagnation: 7/15
Epoch [110/1000] | Train MSE: 0.000106 | Val MSE: 0.000106 | Stagnation: 6/15
Epoch [120/1000] | Train MSE: 0.000105 

100%|██████████| 10000/10000 [00:30<00:00, 325.55it/s]


조기 종료가 활성화된 검증 학습 시작 (Patience: 15)
Epoch [1/1000] | Train MSE: 0.006224 | Val MSE: 0.000682 | Stagnation: 0/15
Epoch [10/1000] | Train MSE: 0.000126 | Val MSE: 0.000117 | Stagnation: 0/15
Epoch [20/1000] | Train MSE: 0.000114 | Val MSE: 0.000107 | Stagnation: 0/15
Epoch [30/1000] | Train MSE: 0.000111 | Val MSE: 0.000105 | Stagnation: 0/15
Epoch [40/1000] | Train MSE: 0.000110 | Val MSE: 0.000104 | Stagnation: 0/15
Epoch [50/1000] | Train MSE: 0.000109 | Val MSE: 0.000103 | Stagnation: 0/15
Epoch [60/1000] | Train MSE: 0.000108 | Val MSE: 0.000103 | Stagnation: 0/15
Epoch [70/1000] | Train MSE: 0.000107 | Val MSE: 0.000102 | Stagnation: 0/15
Epoch [80/1000] | Train MSE: 0.000107 | Val MSE: 0.000102 | Stagnation: 1/15
Epoch [90/1000] | Train MSE: 0.000106 | Val MSE: 0.000102 | Stagnation: 1/15
Epoch [100/1000] | Train MSE: 0.000106 | Val MSE: 0.000101 | Stagnation: 0/15
Epoch [110/1000] | Train MSE: 0.000105 | Val MSE: 0.000101 | Stagnation: 0/15
Epoch [120/1000] | Train MSE: 0.000105 

In [ ]:
loss_proposed = run_strict_verification_experiment(
    feature_list= ['log_kappa', 'v_norm', 'T' , 'torsion','vec_M'],
    train_dir=train_data_dir,
    label_path=train_label_path,
    epochs=1000
)

데이터 및 피처 사전 계산 중...


100%|██████████| 10000/10000 [00:30<00:00, 323.52it/s]


조기 종료가 활성화된 검증 학습 시작 (Patience: 30)
Epoch [1/1000] | Train MSE: 0.002345 | Val MSE: 0.001021 | Stagnation: 0/30
Epoch [10/1000] | Train MSE: 0.000129 | Val MSE: 0.000134 | Stagnation: 0/30
Epoch [20/1000] | Train MSE: 0.000115 | Val MSE: 0.000124 | Stagnation: 0/30
Epoch [30/1000] | Train MSE: 0.000110 | Val MSE: 0.000120 | Stagnation: 0/30
Epoch [40/1000] | Train MSE: 0.000107 | Val MSE: 0.000119 | Stagnation: 2/30
Epoch [50/1000] | Train MSE: 0.000105 | Val MSE: 0.000117 | Stagnation: 0/30
Epoch [60/1000] | Train MSE: 0.000103 | Val MSE: 0.000116 | Stagnation: 0/30
Epoch [70/1000] | Train MSE: 0.000101 | Val MSE: 0.000116 | Stagnation: 1/30
Epoch [80/1000] | Train MSE: 0.000100 | Val MSE: 0.000116 | Stagnation: 5/30
Epoch [90/1000] | Train MSE: 0.000099 | Val MSE: 0.000115 | Stagnation: 0/30
Epoch [100/1000] | Train MSE: 0.000098 | Val MSE: 0.000115 | Stagnation: 2/30
Epoch [110/1000] | Train MSE: 0.000096 | Val MSE: 0.000115 | Stagnation: 12/30
Epoch [120/1000] | Train MSE: 0.000095

In [ ]:
loss_proposed = run_strict_verification_experiment(
    feature_list= ['log_kappa', 'torsion','vec_M'],
    train_dir=train_data_dir,
    label_path=train_label_path,
    epochs=1000,
    patience=100
)

데이터 및 피처 사전 계산 중...


100%|██████████| 10000/10000 [00:30<00:00, 324.93it/s]


조기 종료가 활성화된 검증 학습 시작 (Patience: 100)
Epoch [1/1000] | Train MSE: 0.009276 | Val MSE: 0.001369 | Stagnation: 0/100
Epoch [10/1000] | Train MSE: 0.000141 | Val MSE: 0.000135 | Stagnation: 0/100
Epoch [20/1000] | Train MSE: 0.000122 | Val MSE: 0.000121 | Stagnation: 0/100
Epoch [30/1000] | Train MSE: 0.000116 | Val MSE: 0.000116 | Stagnation: 0/100
Epoch [40/1000] | Train MSE: 0.000113 | Val MSE: 0.000114 | Stagnation: 0/100
Epoch [50/1000] | Train MSE: 0.000110 | Val MSE: 0.000112 | Stagnation: 0/100
Epoch [60/1000] | Train MSE: 0.000109 | Val MSE: 0.000111 | Stagnation: 0/100
Epoch [70/1000] | Train MSE: 0.000107 | Val MSE: 0.000110 | Stagnation: 1/100
Epoch [80/1000] | Train MSE: 0.000106 | Val MSE: 0.000110 | Stagnation: 2/100
Epoch [90/1000] | Train MSE: 0.000105 | Val MSE: 0.000110 | Stagnation: 1/100
Epoch [100/1000] | Train MSE: 0.000105 | Val MSE: 0.000110 | Stagnation: 1/100
Epoch [110/1000] | Train MSE: 0.000104 | Val MSE: 0.000109 | Stagnation: 6/100
Epoch [120/1000] | Train M